Testando BGE-M3

In [28]:
from FlagEmbedding import BGEM3FlagModel
from FlagEmbedding import FlagReranker
from pymongo import MongoClient
from qdrant_client.models import Prefetch, FusionQuery, Fusion, models
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, SparseVectorParams
from qdrant_client.models import PointStruct, SparseVector
import uuid
import pprint

Carregando o modelo BGE-M3

In [3]:
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

Loading weights: 100%|██████████| 391/391 [00:01<00:00, 197.88it/s]


Carregando os dados dos professores do Banco de dados do MongoDB

In [4]:
client = MongoClient("mongodb://localhost:27017/")
db = client["TCC"]
professores = db["Docentes"]

perfis = list(professores.find({}))

Criação de texto somente com o essencial das informações dos professoes (por enquanto só resumo + disciplinas)

In [139]:
def criar_perfil(prof):
    
    nomes_disciplinas = [d["nome"] for d in prof.get("disciplinasSigaa", [])]
    disciplinas_text = ", ".join(nomes_disciplinas)
 
    return (
        f"Nome: {prof.get('nome', '')} "
        f"Disciplinas: {disciplinas_text.lower()}"
        )

In [140]:
professores_texto = []
for professor in perfis:
    perfil_prof = criar_perfil(professor)
    professores_texto.append(perfil_prof)
    
print(professores_texto[0])

Nome: Ana Paula Siqueira Silva de Almeida Disciplinas: engenharia de usabilidade, laboratório de circuitos e eletrônica, projeto robusto de produtos, laboratório de eletrônica analógica 1, engenharia de fatores humanos e usabilidade, introdução à eletrônica analógica, laboratório de introdução à eletrônica analógica, metodologia científica e tecnológica, laboratório de eletrônica analógica i, engenharia cognitiva, eletrônica básica e instrumentação (prática), laboratório de eletrônica digital i, laboratório de eletrônica digital i, tecnologias e fatores humanos, tópicos especiais i: projeto de pesquisa, design thinking para educadores? práticas e soluções para o cotidiano escolar


Realizar o embedding e representação esparsa do texto de cada professor

In [141]:
texto_transformado = model.encode(professores_texto, batch_size=12, max_length=512, return_dense=True, return_sparse=True)

Inference Embeddings: 100%|██████████| 4/4 [01:14<00:00, 18.55s/it]


In [142]:
embeddings_professores = texto_transformado['dense_vecs']
lexical_professores = texto_transformado["lexical_weights"]

In [143]:
perfis = list(professores.find({"nome": {"$exists": True}}))

Adicionando os professores no Qdrant

In [144]:
client_qdrant = QdrantClient(url="http://localhost:6333")

In [145]:
client_qdrant.delete_collection("professores")

True

Criando o banco e adicionando professores

In [146]:
client_qdrant.create_collection(
    collection_name="professores",
    vectors_config={
        "dense": VectorParams(size=1024, distance=Distance.COSINE)},
    sparse_vectors_config={
        "sparse": SparseVectorParams(),
    },
)

True

In [ ]:
points = []
for doc, dense, sparse in zip(perfis, embeddings_professores, lexical_professores):
    sparse_indices = [int(k) for k in sparse.keys()]
    sparse_values = [float(v) for v in sparse.values()]

    points.append(
        PointStruct(
            id=str(uuid.uuid4()),
            vector={
                "dense": dense.tolist(),
                "sparse": SparseVector(indices=sparse_indices, values=sparse_values),
            },
            # metadados
            payload={
                "nome": doc.get("nome"),
                "unidade": doc.get("unidade"),
                "professor_id": str(doc["_id"]),
            }
        )
    )
    
client_qdrant.upsert(collection_name="professores", points=points)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

Simulando interesses de um aluno

In [148]:
aluno_texto = "Compiladores"
aluno = model.encode([aluno_texto.lower()], return_dense=True, return_sparse=True)
embedding_aluno = aluno['dense_vecs'][0]
lexical_aluno = aluno['lexical_weights'][0]

Busca densa

In [149]:
results = client_qdrant.query_points(
    "professores",
    query=embedding_aluno,
    using="dense",
    limit=10,
)

pprint.pp(results.points)

[ScoredPoint(id='0477ae60-1678-4d43-8e97-232e32e7e6ca', version=1, score=0.54002607, payload={'nome': 'Thatyana de Faria Piola Seraphim', 'departamento': None, 'professor_id': '6a90d9afd7c5e34c87791e1e'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id='e3b52c63-2a05-47fc-9e7d-2c3fdd2b52a8', version=1, score=0.4528015, payload={'nome': 'Edmilson Marmo Moreira', 'departamento': None, 'professor_id': '6a90d9afd7c5e34c87791e2f'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id='33b13f0b-626b-454c-a3c4-90f76f5f23ae', version=1, score=0.4424283, payload={'nome': 'Enzo Seraphim', 'departamento': None, 'professor_id': '6a90d9afd7c5e34c87791e3b'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id='bac8f382-914f-42c8-801a-9183e763f642', version=1, score=0.43923247, payload={'nome': 'Carlos Henrique Valério de Moraes', 'departamento': None, 'professor_id': '6a90d9afd7c5e34c87791e2d'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id='0

Busca esparsa

In [150]:
# formato do lexical já em ["Compiladores" : 4.55]

sparse_query = SparseVector(
    indices=[int(k) for k in lexical_aluno.keys()],
    values=[float(v) for v in lexical_aluno.values()],
)

results = client_qdrant.query_points(
    "professores",
    query=sparse_query,
    using="sparse",
    limit=10,
)

pprint.pp(results.points)

[ScoredPoint(id='0477ae60-1678-4d43-8e97-232e32e7e6ca', version=1, score=0.09317435, payload={'nome': 'Thatyana de Faria Piola Seraphim', 'departamento': None, 'professor_id': '6a90d9afd7c5e34c87791e1e'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id='0ae1ada5-cb67-49cc-815b-a75df3b5facb', version=1, score=0.0057592555, payload={'nome': 'Rondineli Rodrigues Pereira', 'departamento': None, 'professor_id': '6a90d9afd7c5e34c87791e32'}, vector=None, shard_key=None, order_value=None)]


In [151]:
sparse_query = SparseVector(
    indices=[int(k) for k in lexical_aluno.keys()],
    values=[float(v) for v in lexical_aluno.values()],
)

prefetch = [
        Prefetch(query=embedding_aluno.tolist(), using="dense", limit=8),
        Prefetch(query=sparse_query, using="sparse", limit=8)
]

results = client_qdrant.query_points(
    collection_name="professores",
    prefetch=prefetch,
    # Busca Hibrida
    query=FusionQuery(fusion=Fusion.RRF),
    limit=5,
)

for point in results.points:
    print(point.payload["nome"], point.score)

Thatyana de Faria Piola Seraphim 1.0
Edmilson Marmo Moreira 0.33333334
Rondineli Rodrigues Pereira 0.33333334
Enzo Seraphim 0.25
Carlos Henrique Valério de Moraes 0.2


In [152]:
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 3311.62it/s]


In [155]:
from bson import ObjectId

In [156]:
pairs = []
for point in results.points:
    teacher = professores.find_one({"_id": ObjectId(point.payload["professor_id"])})
    teacher_text = criar_perfil(teacher)
    pairs.append([aluno_texto, teacher_text])

rerank_scores = reranker.compute_score(pairs, normalize=True)

In [159]:
ranked = sorted(zip(results.points, rerank_scores), key=lambda x: x[1], reverse=True)

for point, score in ranked:
    teacher = professores.find_one({"_id": ObjectId(point.payload["professor_id"])})
    print(f"{teacher['nome']} — score: {score:.4f}")

Thatyana de Faria Piola Seraphim — score: 0.4695
Edmilson Marmo Moreira — score: 0.0035
Enzo Seraphim — score: 0.0021
Carlos Henrique Valério de Moraes — score: 0.0016
Rondineli Rodrigues Pereira — score: 0.0007


Adicionar 

- Fine-tuning
- Avaliar quais informações são relevantes de se obter dos professores
- Que dados serão obtidos dos alunos
- Ajeitar Contextual Retrieval